In [1]:
import time
import mlx.core as mx
from mlx_lm import load, stream_generate
from IPython.display import display, Markdown

print("🔄 Model yuklanmoqda (Qwen 2.5 7B)...")
MODEL_ID = "mlx-community/Qwen2.5-7B-Instruct-4bit"
model, tokenizer = load(MODEL_ID)
print("🚀 Model tayyor! stream_generate orqali KV Cache testi boshlandi.\n")

# Savollar (U1 uzun, keshni tekshirish uchun)
U1 = "Menga sun'iy intellekt tarixi, uning o'rtaga chiqishi, 1956-yilgi Dartmut konferensiyasi va bugungi transformer arxitekturasi inqilobi haqida juda batafsil ma'lumot ber."
U2 = "Rahmat. Endi Alan Tyuring haqida qisqacha yoz."
U3 = "Yaxshi. SI qishlari nima va ular nega sodir bo'lgan?"
U4 = "Transformer modeli nechanchi yili taklif qilingan?"
U5 = "Ajoyib. Hozirgi barcha suhbatimizni 3 ta so'z bilan xulosa qil."

questions = [U1, U2, U3, U4, U5]
system_prompt = {"role": "system", "content": "Siz qisqa javob beruvchi yordamchisiz."}

def run_production_experiment(kv_cache_enabled, trim_enabled):
    history_messages = [system_prompt]
    turn_times = []
    
    # MLX ning stream_generate funksiyasi uchun keshni ushlab turuvchi o'zgaruvchi
    # Agar uni har raundda saqlab qolsak = KV Cache TRUE bo'ladi
    current_cache = None
    
    for idx, user_q in enumerate(questions, 1):
        if trim_enabled:
            # Rejim 2: Triming (Tarix qirqiladi)
            if idx == 1:
                history_messages = [system_prompt, {"role": "user", "content": user_q}]
            elif idx in [2, 3]:
                history_messages.append({"role": "user", "content": user_q})
            else:
                # Turn 4 va 5 da tarix qirqiladi (U1 va uning javobi o'chadi)
                history_messages = [system_prompt] + history_messages[-4:] + [{"role": "user", "content": user_q}]
                # DIQQAT: Trim bo'lgani uchun eski keshni majburlab reset qilamiz
                current_cache = None 
        else:
            # Rejim 1: To'liq tarix
            history_messages.append({"role": "user", "content": user_q})
            
        # Agar KV_Cache = False bo'lsa, har bir raundda keshni nollashtiramiz
        if not kv_cache_enabled:
            current_cache = None
            
        prompt_text = tokenizer.apply_chat_template(history_messages, tokenize=False, add_generation_prompt=True)
        
        start_time = time.time()
        
        # stream_generate orqali tokenlarni olish jarayoni fonda keshni yangilaydi
        # Biz prompt_cache parametri orqali eski keshni uzatamiz
        generator = stream_generate(
            model, 
            tokenizer, 
            prompt=prompt_text, 
            max_tokens=32,
            prompt_cache=current_cache  # Keshni uzatish mexanizmi
        )
        
        # Generatorni oxirigacha aylantirib javobni yig'amiz
        response_text = ""
        for response in generator:
            response_text += response.text
            # Eng asosiysi: yangilangan kesh obyektini generator ichidan sug'urib olamiz!
            if hasattr(response, 'prompt_cache') and kv_cache_enabled:
                current_cache = response.prompt_cache
        
        # Apple Silicon hisob-kitobni to'liq tugatishini kutamiz
        mx.eval(model.parameters())
        duration = time.time() - start_time
        turn_times.append(duration)
        
        history_messages.append({"role": "assistant", "content": response_text})
        
    return turn_times

# To'rtta eksperimentni ishga tushiramiz
print("🔄 Rejim 1 (Cache ON) bajarilmoqda...")
t_r1_on  = run_production_experiment(kv_cache_enabled=True,  trim_enabled=False)

print("🔄 Rejim 1 (Cache OFF) bajarilmoqda...")
t_r1_off = run_production_experiment(kv_cache_enabled=False, trim_enabled=False)

print("🔄 Rejim 2 (Trimmed ON) bajarilmoqda...")
t_r2_on  = run_production_experiment(kv_cache_enabled=True,  trim_enabled=True)

print("🔄 Rejim 2 (Trimmed OFF) bajarilmoqda...")
t_r2_off = run_production_experiment(kv_cache_enabled=False, trim_enabled=True)

# 📊 YAKUNIY JADVALNI CHOP ETISH
markdown_table = f"""
### 📊 REAL PRODUCTION KV CACHE JADVALI (Soniya hisobida)

| Turn | R1 (Cache ON) ⚡ | R1 (Cache OFF) 🐢 | R2 (Cache ON - Trim) ✂️ | R2 (Cache OFF - Trim) |
| :--- | :--- | :--- | :--- | :--- |
| **Turn 1 (U1: Uzun)** | {t_r1_on[0]:.4f} s | {t_r1_off[0]:.4f} s | {t_r2_on[0]:.4f} s | {t_r2_off[0]:.4f} s |
| **Turn 2 (U2: Qisqa)** | **{t_r1_on[1]:.4f} s** 🔥 | {t_r1_off[1]:.4f} s | **{t_r2_on[1]:.4f} s** 🔥 | {t_r2_off[1]:.4f} s |
| **Turn 3 (U3: Qisqa)** | **{t_r1_on[2]:.4f} s** 🔥 | {t_r1_off[2]:.4f} s | **{t_r2_on[2]:.4f} s** 🔥 | {t_r2_off[2]:.4f} s |
| **Turn 4 (U4: Trim!)** | **{t_r1_on[3]:.4f} s** 🔥 | {t_r1_off[3]:.4f} s | {t_r2_on[3]:.4f} s 🛑 *(Sakradi)* | {t_r2_off[3]:.4f} s |
| **Turn 5 (U5: Qisqa)** | **{t_r1_on[4]:.4f} s** 🔥 | {t_r1_off[4]:.4f} s | **{t_r2_on[4]:.4f} s** 🔥 | {t_r2_off[4]:.4f} s |
"""

display(Markdown(markdown_table))

/Users/rakhmonberdiev/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/rakhmonberdiev/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔄 Model yuklanmoqda (Qwen 2.5 7B)...


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 131988.59it/s]


🚀 Model tayyor! stream_generate orqali KV Cache testi boshlandi.

🔄 Rejim 1 (Cache ON) bajarilmoqda...
🔄 Rejim 1 (Cache OFF) bajarilmoqda...
🔄 Rejim 2 (Trimmed ON) bajarilmoqda...
🔄 Rejim 2 (Trimmed OFF) bajarilmoqda...



### 📊 REAL PRODUCTION KV CACHE JADVALI (Soniya hisobida)

| Turn | R1 (Cache ON) ⚡ | R1 (Cache OFF) 🐢 | R2 (Cache ON - Trim) ✂️ | R2 (Cache OFF - Trim) |
| :--- | :--- | :--- | :--- | :--- |
| **Turn 1 (U1: Uzun)** | 0.9476 s | 0.8521 s | 0.8551 s | 0.8535 s |
| **Turn 2 (U2: Qisqa)** | **1.0188 s** 🔥 | 0.9991 s | **1.0042 s** 🔥 | 1.0136 s |
| **Turn 3 (U3: Qisqa)** | **1.1583 s** 🔥 | 1.1687 s | **1.1465 s** 🔥 | 1.1504 s |
| **Turn 4 (U4: Trim!)** | **1.2922 s** 🔥 | 1.3071 s | 1.0826 s 🛑 *(Sakradi)* | 1.0726 s |
| **Turn 5 (U5: Qisqa)** | **1.4489 s** 🔥 | 1.4469 s | **0.7794 s** 🔥 | 0.7596 s |


In [2]:
import mlx_lm
import mlx_lm.utils as utils

print("--- MLX_LM ichidagi barcha funksiya va klasslar ---")
print([x for x in dir(mlx_lm) if "cache" in x.lower() or "generate" in x.lower()])

print("\n--- MLX_LM.UTILS ichidagi barcha funksiya va klasslar ---")
print([x for x in dir(utils) if "cache" in x.lower() or "generate" in x.lower()])

--- MLX_LM ichidagi barcha funksiya va klasslar ---
['__cached__', 'batch_generate', 'generate', 'stream_generate']

--- MLX_LM.UTILS ichidagi barcha funksiya va klasslar ---
['__cached__']


In [3]:
import time
import mlx.core as mx
from mlx_lm import load, generate
from IPython.display import display, Markdown

print("🔄 Model yuklanmoqda...")
MODEL_ID = "mlx-community/Qwen2.5-7B-Instruct-4bit"
model, tokenizer = load(MODEL_ID)
print("🚀 Model tayyor! Ro'yxat asosidagi KV Cache testi boshlandi.\n")

U1 = "Menga sun'iy intellekt tarixi, uning o'rtaga chiqishi, 1956-yilgi Dartmut konferensiyasi va bugungi transformer arxitekturasi inqilobi haqida juda batafsil ma'lumot ber."
U2 = "Rahmat. Endi Alan Tyuring haqida qisqacha yoz."
U3 = "Yaxshi. SI qishlari nima va ular nega sodir bo'lgan?"
U4 = "Transformer modeli nechanchi yili taklif qilingan?"
U5 = "Ajoyib. Hozirgi barcha suhbatimizni 3 ta so'z bilan xulosa qil."
questions = [U1, U2, U3, U4, U5]
system_prompt = {"role": "system", "content": "Siz qisqa javob beruvchi yordamchisiz."}

def run_pure_list_cache_experiment(kv_cache_enabled, trim_enabled):
    history_messages = [system_prompt]
    turn_times = []
    
    # Kesh obyekti sifatida bo'sh ro'yxat ochamiz. 
    # MLX ning generate funksiyasi bu ro'yxatni ichkaridan mutate qiladi.
    session_cache = [] if kv_cache_enabled else None
    
    for idx, user_q in enumerate(questions, 1):
        if trim_enabled:
            if idx == 1:
                history_messages = [system_prompt, {"role": "user", "content": user_q}]
            elif idx in [2, 3]:
                history_messages.append({"role": "user", "content": user_q})
            else:
                # Turn 4 va 5 da tarix qirqiladi
                history_messages = [system_prompt] + history_messages[-4:] + [{"role": "user", "content": user_q}]
                # DIQQAT: Kontekst o'zgargani (trim bo'lgani) sababli kesh ro'yxatini tozalaymiz!
                if kv_cache_enabled:
                    session_cache = []
        else:
            history_messages.append({"role": "user", "content": user_q})
            
        # Agar KV_Cache o'chiq bo'lsa, har safar yangi bo'sh ro'yxat beramiz (yoki None)
        if not kv_cache_enabled:
            session_cache = []
            
        prompt_text = tokenizer.apply_chat_template(history_messages, tokenize=False, add_generation_prompt=True)
        
        start_time = time.time()
        
        # prompt_cache ga ro'yxatni beramiz. MLX uni o'zi keshga aylantirib oladi!
        response_text = generate(
            model, 
            tokenizer, 
            prompt=prompt_text, 
            max_tokens=32, 
            prompt_cache=session_cache,  
            verbose=False
        )
        
        mx.eval(model.parameters()) 
        duration = time.time() - start_time
        turn_times.append(duration)
        
        history_messages.append({"role": "assistant", "content": response_text})
        
    return turn_times

# Tajribalarni boshlaymiz
print("🔄 R1 (Cache ON) bajarilmoqda...")
t_r1_on  = run_pure_list_cache_experiment(kv_cache_enabled=True,  trim_enabled=False)

print("🔄 R1 (Cache OFF) bajarilmoqda...")
t_r1_off = run_pure_list_cache_experiment(kv_cache_enabled=False, trim_enabled=False)

print("🔄 R2 (Trimmed ON) bajarilmoqda...")
t_r2_on  = run_pure_list_cache_experiment(kv_cache_enabled=True,  trim_enabled=True)

print("🔄 R2 (Trimmed OFF) bajarilmoqda...")
t_r2_off = run_pure_list_cache_experiment(kv_cache_enabled=False, trim_enabled=True)

# Jadvalni chiqarish
markdown_table = f"""
### 📊 YAKUNIY VA TO'G'RI KV CACHE JADVALI (Soniya hisobida)

| Turn | R1 (Cache ON) ⚡ | R1 (Cache OFF) 🐢 | R2 (Cache ON - Trim) ✂️ | R2 (Cache OFF - Trim) |
| :--- | :--- | :--- | :--- | :--- |
| **Turn 1 (U1: Uzun)** | {t_r1_on[0]:.4f} s | {t_r1_off[0]:.4f} s | {t_r2_on[0]:.4f} s | {t_r2_off[0]:.4f} s |
| **Turn 2 (U2: Qisqa)** | **{t_r1_on[1]:.4f} s** 🚀 | {t_r1_off[1]:.4f} s | **{t_r2_on[1]:.4f} s** 🚀 | {t_r2_off[1]:.4f} s |
| **Turn 3 (U3: Qisqa)** | **{t_r1_on[2]:.4f} s** 🚀 | {t_r1_off[2]:.4f} s | **{t_r2_on[2]:.4f} s** 🚀 | {t_r2_off[2]:.4f} s |
| **Turn 4 (U4: Trim!)** | **{t_r1_on[3]:.4f} s** 🚀 | {t_r1_off[3]:.4f} s | {t_r2_on[3]:.4f} s 🛑 *(Sakradi)* | {t_r2_off[3]:.4f} s |
| **Turn 5 (U5: Qisqa)** | **{t_r1_on[4]:.4f} s** 🚀 | {t_r1_off[4]:.4f} s | **{t_r2_on[4]:.4f} s** 🚀 | {t_r2_off[4]:.4f} s |
"""
display(Markdown(markdown_table))

🔄 Model yuklanmoqda...


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 114044.52it/s]


🚀 Model tayyor! Ro'yxat asosidagi KV Cache testi boshlandi.

🔄 R1 (Cache ON) bajarilmoqda...


IndexError: list index out of range

xlm_lm.generate() da kv_cache mavjud emas
